# Qualitative comparison — Sentinel-2 gap-filling adaptation levels

Displays gap-filling reconstructions on randomly drawn samples from the
Sentinel-2 gap-filling evaluation dataset, side by side, for all four
adaptation levels (zero-shot, fine-tuning, structured masking, from-scratch
training) plus the two non-learned interpolation baselines (Ordinary Kriging,
linear interpolation). Each sample's dataset row index is shown so it can be
found again. Panels show a true-colour (B4/B3/B2) composite with a per-row
robust 2–98 percentile stretch; synthetic gap pixels are rendered black.

Workflow:
1. Run the **imports**, **config**, and **load dataset + models** cells once.
2. Re-run the **browse** cell repeatedly (changing `seed`, or setting
   `sample_indices` directly) to page through random draws.
3. Use the **save** cell to export a page you like as a PNG.

In [ ]:
import os
import sys

import numpy as np
import torch
from joblib import Parallel, delayed
from matplotlib import pyplot as plt
from pykrige.ok import OrdinaryKriging
from scipy.interpolate import griddata
from scipy.spatial import QhullError

_gap_aware_root = os.environ.get(
    "GAP_AWARE_ROOT",
    os.path.abspath(os.path.join(os.path.dirname("__file__"), "..", "..", "..")),
)
if _gap_aware_root not in sys.path:
    sys.path.insert(0, _gap_aware_root)

from model_gap_aware.maskfeat import MaskFeatGapFiller
from sentinel.src.evaluation.evaluate_gapfilling import (
    GapFillingEvalDataset,
    find_checkpoint,
)

In [ ]:
# --- Dataset ---
_s2_data_dir = os.environ.get(
    "S2_DATA_DIR", os.path.join(_gap_aware_root, "sentinel", "data")
)
gap_fill_root = os.path.join(_s2_data_dir, "gap_filling")
device = "cpu"
normalize = True   # ÷10000 + clip [0, 1]; must match the normalization used during training/eval

# --- Checkpoint discovery ---
# One log_dir per adaptation level, matching each exp0N/config.yaml's training.log_dir.
LOG_DIRS = {
    "zero_shot": os.path.join(_gap_aware_root, "sentinel/lightning_logs/experiments/exp01_zero_shot"),
    "finetuning": os.path.join(_gap_aware_root, "sentinel/lightning_logs/experiments/exp02_finetuning"),
    "structured_masking": os.path.join(_gap_aware_root, "sentinel/lightning_logs/experiments/exp03_structured_masking"),
    "from_scratch": os.path.join(_gap_aware_root, "sentinel/lightning_logs/experiments/exp04_scratch_gapfilling"),
}

# Manual override: set a checkpoint path for a level to skip auto-discovery
# for it; leave None to auto-discover the lowest-MAE_val checkpoint under
# that level's log_dir via find_checkpoint() (same logic evaluate_gapfilling.sh
# uses).
MANUAL_CHECKPOINTS = {
    "zero_shot": None,
    "finetuning": os.path.join(_gap_aware_root, "sentinel/lightning_logs/experiments/exp02_finetuning/sample_ablation/n0400/version_0/checkpoints/gapfiller-epoch=13-MAE_val=0.0153.ckpt"),
    "structured_masking": None,
    "from_scratch": os.path.join(_gap_aware_root, "sentinel/lightning_logs/experiments/exp04_scratch_gapfilling/version_3/checkpoints/gapfiller-epoch=48-MAE_val=0.0147.ckpt"),
}

# MANUAL_CHECKPOINTS = {
#     "zero_shot": os.path.join(_gap_aware_root, "sentinel/lightning_logs/experiments/exp01_zero_shot/version_1/checkpoints/gapfiller-epoch=00-MAE_val=0.2018.ckpt"),
#     "finetuning": os.path.join(_gap_aware_root, "sentinel/lightning_logs/experiments/exp02_finetuning/version_2/checkpoints/gapfiller-epoch=00-MAE_val=0.0910.ckpt"),
#     "structured_masking": os.path.join(_gap_aware_root, "sentinel/lightning_logs/experiments/exp03_structured_masking/version_1/checkpoints/gapfiller-epoch=00-MAE_val=0.2010.ckpt"),
#     "from_scratch": os.path.join(_gap_aware_root, "sentinel/lightning_logs/experiments/exp04_scratch_gapfilling/version_1/checkpoints/gapfiller-epoch=00-MAE_val=0.1229.ckpt"),
# }

# --- Interpolation baselines ---
variogram_model = "linear"   # pykrige variogram model, matches interpolation_baseline/config.yaml
n_jobs = 1                    # joblib workers for kriging/linear fits (one fit per displayed sample)
max_obs_points = 1000        # cap on observed pixels per band's Kriging fit (subsampled if exceeded)
kriging_seed = 42            # rng seed for the Kriging observed-pixel subsample

# --- RGB display / MAE units ---
# Sentinel-2 has no single physical unit like ESDC's Kelvin (10 bands, each its
# own reflectance scale), so MAE stays in normalized [0, 1] units. For display
# only, values are pushed back to raw DN space before the per-row percentile
# stretch.
DN_SCALE = 10000.0

In [ ]:
dataset = GapFillingEvalDataset(gap_fill_root, normalize=normalize)
print(f"Dataset size: {len(dataset)} samples")

models = {}
for level, log_dir in LOG_DIRS.items():
    ckpt_path = find_checkpoint(log_dir, MANUAL_CHECKPOINTS[level])
    print(f"[{level}] {ckpt_path}")
    m = MaskFeatGapFiller.load_from_checkpoint(ckpt_path, map_location=device)
    m.to(device)
    m.eval()
    models[level] = m

In [ ]:
def _mean_band_mae(pred_chw, gt_chw, valid_chw):
    """Per-band mean |pred - gt| over (synthetic gap AND valid-GT) pixels, then
    the mean over bands, in normalized [0, 1] units. Mirrors
    evaluate_gapfilling.py's per-band accumulate-then-average convention."""
    per_band = []
    for c in range(pred_chw.shape[0]):
        m = valid_chw[c]
        per_band.append(
            float(np.abs(pred_chw[c][m] - gt_chw[c][m]).mean()) if m.any() else np.nan
        )
    return float(np.nanmean(per_band)) if np.any(np.isfinite(per_band)) else float("nan")


def krige_predict(x, gt_middle, gap_mask, variogram_model, max_obs_points, rng):
    """Mirrors evaluate_kriging_baseline.py::_krige_band's fit/predict, per band,
    but returns the full reconstructed (C, H, W) frame (observed pixels kept,
    gap pixels filled) instead of an aggregated error."""
    mid = np.asarray(x[2])          # (C, H, W) middle input frame, NaN at gaps
    gt = np.asarray(gt_middle)      # (C, H, W)
    mask = np.asarray(gap_mask)     # (H, W), 1 = gap

    pred_full = mid.copy()
    for c in range(mid.shape[0]):
        mid_b, gt_b = mid[c], gt[c]
        obs_rows, obs_cols = np.where(~np.isnan(mid_b))
        gap_rows, gap_cols = np.where((mask > 0) & ~np.isnan(gt_b))
        if len(gap_rows) == 0 or len(obs_rows) < 3:
            continue
        if len(obs_rows) > max_obs_points:
            keep = rng.choice(len(obs_rows), size=max_obs_points, replace=False)
            obs_rows, obs_cols = obs_rows[keep], obs_cols[keep]
        obs_values = mid_b[obs_rows, obs_cols]
        if np.ptp(obs_values) == 0:
            # Zero-variance neighbourhood: pykrige's variogram fit degenerates;
            # the Ordinary Kriging prediction is just the constant observed value.
            pred = np.full(len(gap_rows), obs_values[0])
        else:
            ok = OrdinaryKriging(
                obs_cols.astype(float), obs_rows.astype(float), obs_values,
                variogram_model=variogram_model, verbose=False, enable_plotting=False,
            )
            pred, _ = ok.execute("points", gap_cols.astype(float), gap_rows.astype(float))
            pred = np.asarray(pred)
        pred_full[c, gap_rows, gap_cols] = pred
    return pred_full


def linear_interp_predict(x, gt_middle, gap_mask):
    """Mirrors evaluate_linear_interpolation_baseline.py::_linear_interp_band's
    fit/predict, per band, but returns the full reconstructed (C, H, W) frame."""
    mid = np.asarray(x[2])
    gt = np.asarray(gt_middle)
    mask = np.asarray(gap_mask)

    pred_full = mid.copy()
    for c in range(mid.shape[0]):
        mid_b, gt_b = mid[c], gt[c]
        obs_rows, obs_cols = np.where(~np.isnan(mid_b))
        gap_rows, gap_cols = np.where((mask > 0) & ~np.isnan(gt_b))
        if len(gap_rows) == 0 or len(obs_rows) < 3:
            continue
        obs_points = np.column_stack((obs_cols, obs_rows)).astype(float)
        obs_values = mid_b[obs_rows, obs_cols]
        gap_points = np.column_stack((gap_cols, gap_rows)).astype(float)
        try:
            pred = griddata(obs_points, obs_values, gap_points, method="linear")
        except QhullError:
            pred = np.full(len(gap_points), np.nan)
        nan_mask = np.isnan(pred)
        if nan_mask.any():
            pred[nan_mask] = griddata(
                obs_points, obs_values, gap_points[nan_mask], method="nearest"
            )
        pred_full[c, gap_rows, gap_cols] = pred
    return pred_full


def run_nn_methods(idx):
    """Run all four adaptation-level models on dataset[idx]. Returns a dict
    with the sample's normalized fields and per-method predictions/MAE (mean
    over the 10 bands, in normalized [0, 1] units, over gap pixels with valid
    ground truth)."""
    input_series, gt_middle, gap_mask = dataset[idx]   # numpy: (T,C,H,W), (C,H,W), (H,W)
    mid_series = input_series[2]                       # (C, H, W), NaN at gap pixels
    gap_mask_np = np.asarray(gap_mask) > 0.5           # (H, W)
    gt_middle = np.asarray(gt_middle)
    # gap pixels with valid (non-NaN) ground truth, per band
    valid = gap_mask_np[None, :, :] & ~np.isnan(gt_middle)          # (C, H, W)
    # full clean middle frame = observed pixels + gap-pixel ground truth
    gt_full_norm = np.where(gap_mask_np[None, :, :], gt_middle, mid_series).astype(np.float32)

    preds, maes = {}, {}
    x_in = torch.from_numpy(input_series).unsqueeze(0).to(device)   # (1, T, C, H, W)
    with torch.no_grad():
        for level, model in models.items():
            pred, _ = model(x_in, gap_only_masking=True)            # (1, C, H, W)
            pred_np = pred[0].cpu().numpy()
            preds[level] = np.where(gap_mask_np[None, :, :], pred_np, mid_series).astype(np.float32)
            maes[level] = _mean_band_mae(pred_np, gt_middle, valid)

    return dict(
        idx=idx, input_series=input_series, gt_middle=gt_middle, gap_mask=gap_mask,
        mid_series=mid_series, gt_full_norm=gt_full_norm,
        gap_mask_np=gap_mask_np, valid=valid, preds=preds, maes=maes,
    )


def add_baseline_predictions(results_list, variogram_model, n_jobs, max_obs_points, seed):
    """Fits Kriging + linear interpolation for each sample in results_list
    (in place) and adds their full-frame predictions/MAE alongside the NN
    methods'."""

    def _fit_one(r):
        rng = np.random.default_rng(seed + r["idx"])
        kriging_full = krige_predict(
            r["input_series"], r["gt_middle"], r["gap_mask"],
            variogram_model, max_obs_points, rng,
        )
        linear_full = linear_interp_predict(r["input_series"], r["gt_middle"], r["gap_mask"])
        return kriging_full, linear_full

    fitted = Parallel(n_jobs=n_jobs)(delayed(_fit_one)(r) for r in results_list)

    for r, (kriging_full, linear_full) in zip(results_list, fitted):
        for name, pred_full in (("kriging", kriging_full), ("linear", linear_full)):
            r["preds"][name] = pred_full.astype(np.float32)
            r["maes"][name] = _mean_band_mae(pred_full, r["gt_middle"], r["valid"])
    return results_list


# --- Display ----------------------------------------------------------------
ALL_METHODS = ["zero_shot", "finetuning", "structured_masking", "from_scratch", "kriging", "linear"]
DISPLAY_NAMES = {
    "zero_shot": "Zero-shot",
    "finetuning": "Fine-tuning",
    "structured_masking": "Structured masking",
    "from_scratch": "From-scratch",
    "kriging": "Kriging",
    "linear": "Linear interp.",
}

N_TIMESTEPS = 5
MIDDLE_T = N_TIMESTEPS // 2  # 2

TIMESTEP_TITLES = [f"t={t}" if t != MIDDLE_T else f"t={t} (gappy)" for t in range(N_TIMESTEPS)]
PRED_ROW_TITLES = ["Ground truth"] + [DISPLAY_NAMES[m] for m in ALL_METHODS]

# Column layout for the single-row-per-sample figure:
#   [N_TIMESTEPS input frames] [spacer] [ground truth] [spacer] [len(ALL_METHODS) results]
# The spacer columns hold NO axes -- their width alone sets the gap between the
# three groups. PANEL_WSPACE sets the (smaller) gap between adjacent panels.
SPACER_RATIO = 0.06   # width of each blank spacer column, relative to an image column
PANEL_WSPACE = 0.035  # gap between adjacent image panels, as a fraction of panel width
COL_WIDTH_RATIOS = (
    [1.0] * N_TIMESTEPS
    + [SPACER_RATIO, 1.0, SPACER_RATIO]
    + [1.0] * len(ALL_METHODS)
)
N_COLS = len(COL_WIDTH_RATIOS)                 # 14
GT_COL = N_TIMESTEPS + 1                       # 6
FIRST_METHOD_COL = N_TIMESTEPS + 3             # 8

RGB_CHANNELS = [2, 1, 0]   # array channels: B4 (Red), B3 (Green), B2 (Blue)


def robust_rgb(b4, b3, b2, pmin=2, pmax=98, lo=None, hi=None):
    """(H, W) bands -> ((H, W, 3) RGB, (H, W) gap mask) via a linear pmin/pmax
    percentile stretch. Verbatim from
    sentinel/src/visualization/visualize_gapfilling_eval.ipynb. Gap pixels
    (NaN in any band) are rendered black."""
    rgb = np.stack([b4, b3, b2], axis=-1).astype(np.float32)
    gaps = np.any(np.isnan(rgb), axis=-1)
    if lo is None or hi is None:
        lo = np.nanpercentile(rgb, pmin)
        hi = np.nanpercentile(rgb, pmax)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = 0.0, 2500.0
    rgb = (rgb - lo) / (hi - lo + 1e-6)
    rgb = np.clip(rgb, 0, 1)
    rgb[gaps] = 0.0
    return rgb, gaps


def frame_to_rgb(arr_chw, lo=None, hi=None, dn_scale=DN_SCALE):
    """(C, H, W) normalized reflectance -> (H, W, 3) display RGB. NumPy port of
    tensor_to_rgb from visualize_gapfilling_eval.ipynb."""
    arr = np.asarray(arr_chw)
    r, g, b = RGB_CHANNELS
    rgb, _ = robust_rgb(arr[r] * dn_scale, arr[g] * dn_scale, arr[b] * dn_scale, lo=lo, hi=hi)
    return rgb


def row_lo_hi(r, dn_scale=DN_SCALE):
    """Joint 2/98 percentile (raw-DN space) over the RGB bands of the 5 input
    frames + ground truth for one sample; every panel in that row -- including
    the predictions -- is then stretched with this shared lo/hi (mirrors the
    per-row stretch in visualize_gapfilling_eval.ipynb). Predictions are
    deliberately excluded from the percentile estimate so an out-of-range
    model output cannot skew the scene's contrast."""
    frames = (
        [r["input_series"][t] for t in range(N_TIMESTEPS)]
        + [r["gt_full_norm"]]
    )
    vals = []
    for f in frames:
        f = np.asarray(f)
        for c in RGB_CHANNELS:
            band = f[c] * dn_scale
            v = band[np.isfinite(band)]
            if v.size:
                vals.append(v)
    if not vals:
        return 0.0, 2500.0
    combined = np.concatenate(vals)
    lo, hi = float(np.percentile(combined, 2)), float(np.percentile(combined, 98))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return 0.0, 2500.0
    return lo, hi

## Browse

Re-run this cell to page through new random draws (change `seed`), or pin
specific dataset row indices via `sample_indices` (e.g. `sample_indices =
[3, 7]`). The page always shows `n_samples` rows: pinned indices come
first, and any remaining rows are filled with random draws. Set
`sample_indices = None` for a fully random page.

In [ ]:
seed = None            # change this to get a new random draw
n_samples = 6        # samples per page -- the page is always filled to this many
# sample_indices = None  # specific dataset row indices to pin; remaining slots are filled at random
sample_indices = [5725, 520, 739, 322, 5284, 125]

rng = np.random.default_rng(seed)

requested = [] if sample_indices is None else list(dict.fromkeys(sample_indices))
n_fill = max(0, n_samples - len(requested))
if n_fill:
    pool = np.setdiff1d(np.arange(len(dataset)), requested)
    if len(pool) < n_fill:
        print(f"Warning: only {len(pool)} sample(s) available -- showing all of them.")
    fill = rng.choice(pool, size=min(n_fill, len(pool)), replace=False)
    sample_indices = requested + sorted(fill.tolist())
else:
    sample_indices = requested

print(f"Showing {len(sample_indices)} samples: {sample_indices}")

results = [run_nn_methods(idx) for idx in sample_indices]
results = add_baseline_predictions(results, variogram_model, n_jobs, max_obs_points, kriging_seed)

# One row per sample: [5 input frames] | spacer | [ground truth] | spacer | [6 results].
# Axes are created only in the real columns; the two spacer columns stay empty so
# their width alone drives the larger inter-group gap.
fig = plt.figure(
    figsize=(2.6 * (N_TIMESTEPS + 1 + len(ALL_METHODS)) + 1.0, 2.9 * len(results)),
    layout="constrained",
)
fig.get_layout_engine().set(w_pad=0.02, h_pad=0.03, wspace=PANEL_WSPACE, hspace=0.03)
gs = fig.add_gridspec(len(results), N_COLS, width_ratios=COL_WIDTH_RATIOS)

for i, r in enumerate(results):
    row_lo, row_hi = row_lo_hi(r)
    gt_rgb = frame_to_rgb(r["gt_full_norm"], lo=row_lo, hi=row_hi)

    # --- input frames (first N_TIMESTEPS columns) ---
    for t in range(N_TIMESTEPS):
        ax = fig.add_subplot(gs[i, t])
        ax.imshow(frame_to_rgb(r["input_series"][t], lo=row_lo, hi=row_hi))
        ax.set_xticks([])
        ax.set_yticks([])
        if t == 0:
            ax.set_ylabel(f"idx={r['idx']}", fontsize=11, fontweight="bold")
        if i == 0:
            ax.set_title(TIMESTEP_TITLES[t])

    # --- ground truth ---
    ax = fig.add_subplot(gs[i, GT_COL])
    ax.imshow(gt_rgb)
    ax.set_xticks([])
    ax.set_yticks([])
    if i == 0:
        ax.set_title(PRED_ROW_TITLES[0])

    # --- gap-filling results ---
    for j, method in enumerate(ALL_METHODS):
        ax = fig.add_subplot(gs[i, FIRST_METHOD_COL + j])
        ax.imshow(frame_to_rgb(r["preds"][method], lo=row_lo, hi=row_hi))
        ax.set_xticks([])
        ax.set_yticks([])
        mae = r["maes"][method]
        mae_str = f"MAE={mae:.4f}" if not np.isnan(mae) else "MAE=n/a"
        ax.set_title(
            f"{PRED_ROW_TITLES[1 + j]}\n{mae_str}" if i == 0 else mae_str,
            fontsize=9,
        )

fig.suptitle(
    "Sentinel-2 gap-filling: qualitative comparison across adaptation levels",
    fontsize=13,
)
plt.show()

## Save

Export the current page's figure as a PNG once you've found a good set of
samples.

In [ ]:
def save_current_figure(fig, sample_indices, out_dir=None):
    if out_dir is None:
        out_dir = os.path.join(_gap_aware_root, "sentinel", "experiments", "qualitative", "figures")
    os.makedirs(out_dir, exist_ok=True)
    fname = "sample_" + "_".join(str(i) for i in sample_indices) + ".png"
    out_path = os.path.join(out_dir, fname)
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    print(f"Saved: {out_path}")
    return out_path


save_current_figure(fig, sample_indices)